# DARUAN: Quantum Variational Activation in CUDA-Q

This notebook implements **DARUAN** (*DatA Re-Uploading ActivatioN*), the
single-qubit quantum variational activation function (QVAF) introduced in
[Jiang et al., arXiv:2509.14026](https://arxiv.org/abs/2509.14026).

DARUAN is a classically simulable, NISQ-friendly circuit that maps a scalar
input $x$ to an expectation value $\phi(x)=\langle Z\rangle$. In the paper it
serves as an edge activation inside quantum-inspired Kolmogorov–Arnold networks
(QKANs). This application keeps the example small: **one qubit**, a few
re-uploads, and a classical 1D function-fitting loop around `cudaq.observe`.

The goal is to show how a data-reuploading variational circuit can act as a
learnable nonlinearity that remains executable on CUDA-Q simulator and hardware
targets.

## Circuit

With $r$ re-uploads (we use $r=3$):

$$
U(x)=W^{(r+1)}\prod_{\ell=r}^{1}\Bigl[S(w_\ell x+b_\ell)\,W^{(\ell)}\Bigr],
\qquad
W^{(\ell)}=R_z(a_\ell)\,R_y(\pi/3)\,R_z(b_\ell),
\qquad
S(\varphi)=R_z(\varphi).
$$

The activation is $\phi(x)=\langle 0|U^\dagger(x)\,Z\,U(x)|0\rangle$.

In [ ]:
import math

import cudaq
import matplotlib.pyplot as plt
import numpy as np
from cudaq import spin
from scipy.optimize import minimize

cudaq.set_target("qpp-cpu")  # use "nvidia" on GPU hosts
print("target:", cudaq.get_target().name)

REPS = 3
N_PARAMS = 4 * REPS + 2  # 14

In [ ]:
@cudaq.kernel
def daruan_kernel(x: float, params: list[float]):
    """Single-qubit DARUAN with REPS=3 re-uploads."""
    q = cudaq.qubit()
    # Layer 0
    rz(params[0], q)
    ry(math.pi / 3.0, q)
    rz(params[1], q)
    rz(params[2] * x + params[3], q)
    # Layer 1
    rz(params[4], q)
    ry(math.pi / 3.0, q)
    rz(params[5], q)
    rz(params[6] * x + params[7], q)
    # Layer 2
    rz(params[8], q)
    ry(math.pi / 3.0, q)
    rz(params[9], q)
    rz(params[10] * x + params[11], q)
    # Final W^(r+1)
    rz(params[12], q)
    ry(math.pi / 3.0, q)
    rz(params[13], q)


def phi(x: float, params) -> float:
    return float(
        cudaq.observe(daruan_kernel, spin.z(0), float(x),
                      list(map(float, params))).expectation())


# Smoke test
p0 = np.zeros(N_PARAMS)
for ell in range(REPS):
    p0[4 * ell + 2] = 2.0**ell
print("φ(0.3) =", phi(0.3, p0))

## 1D function fitting

Fit $f(x)=\sin(4x)$ on $[-1,1]$ by minimizing MSE of $\phi(x)$ vs. $f(x)$.
The classical host uses SciPy (`COBYLA` warm-start + `L-BFGS-B`), the same
pattern as CUDA-Q's optimizer examples.

In [ ]:
SEED = 0
rng = np.random.default_rng(SEED)

xs = rng.uniform(-1.0, 1.0, size=40)
ys = np.sin(4.0 * xs)

params0 = rng.normal(0.0, 0.3, size=N_PARAMS)
for ell in range(REPS):
    params0[4 * ell + 2] = float(2**ell)
    params0[4 * ell + 3] = 0.0

history = []


def mse(theta):
    preds = np.array([phi(float(x), theta) for x in xs])
    loss = float(np.mean((preds - ys) ** 2))
    history.append(loss)
    return loss


print("initial MSE", mse(params0))
res = minimize(mse, params0, method="COBYLA",
               options={"maxiter": 50, "rhobeg": 0.5})
res = minimize(mse, res.x, method="L-BFGS-B",
               options={"maxiter": 100, "ftol": 1e-12})
params = res.x
print("final MSE  ", float(res.fun), "success=", res.success)

In [ ]:
grid = np.linspace(-1.0, 1.0, 120)
target = np.sin(4.0 * grid)
pred = np.array([phi(float(x), params) for x in grid])

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].plot(grid, target, label="sin(4x)", lw=2)
axes[0].plot(grid, pred, "--", label=r"DARUAN $\phi(x)$", lw=2)
axes[0].scatter(xs, ys, s=12, alpha=0.5, label="train")
axes[0].set_xlabel("x")
axes[0].legend()
axes[0].set_title("Learned activation")

axes[1].semilogy(history, lw=2)
axes[1].set_xlabel("objective calls")
axes[1].set_ylabel("MSE")
axes[1].set_title("Training")
fig.tight_layout()
plt.show()

print("grid MSE", float(np.mean((pred - target) ** 2)))

## Next steps

- Increase `REPS` to enlarge the accessible Fourier support (see Theorem 2.2 in
  the paper).
- Swap `qpp-cpu` for `nvidia` on GPU hosts, or a hardware provider target for
  inference validation of a trained activation.
- Embed many independent DARUAN kernels as KAN edge activations for a full
  quantum-inspired KAN — this notebook intentionally stops at a single
  activation plus a 1D fit.

**Reference:** Jiang, Huang, Chen, Goan, *Quantum Variational Activation
Functions Empower Kolmogorov-Arnold Networks*,
[arXiv:2509.14026](https://arxiv.org/abs/2509.14026).